<a href="https://colab.research.google.com/github/joyashre/LLM-NLP-Learning-task/blob/main/NLPAssignment_TAsk_1_joyi.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Task 1

**QUESTION:** Make a small set of English sentences (say 10 sentences), and perform the following tasks.

1. Implement either of the following tokenization: (a) BPE (b) Wordpiece.

2. Perform stop word removal.

3. Perform stemming and lemmatization on each word.

4. Build three N-gram Language models (A) Unigram, (B) Bigram (C) Trigram. Here your task would be the next word prediction.
5. Implement an n-gram spelling correction model where n = 2 (hence bigram).

> 5.1.  Do not implement character level n-gram models.

> 5.2.  Implement a word level n-gram model.

> 5.3. Use your small set of English sentences to build the vocabulary.

> 5.4. Take a new sentence from the user.

>5.5. Iterate over each word of that user sentence. For each selected word, get a candidate word from vocabulary with least levenshtein distance.

> 5.6. Use bigram language model to decide which word is more likely ... selected word vs candidate word.

> 5.7. The word with more probability would be chosen as the final word.

In [ ]:
!pip install transformers nltk

In [ ]:
import nltk
from transformers import AutoTokenizer # Huggingface se pre-trained tokenizers load karne ka smart tarika hai. Hum ise WordPiece tokenization ke liye use karenge.
from nltk.corpus import stopwords #stopwords - remove because less meaning :"the", "an", "a", "in"
from nltk.stem import PorterStemmer, WordNetLemmatizer # WordNetLemmatizer: reduce words to their base or dictionary form

In [ ]:
# Download required NLTK data dictionaries behind the scenes
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4') # Open Multilingual WordNet. The 1.4 simply refers to a specific, highly stable version of this dataset that NLTK relies on.

# NLTK install karte ho, toh wo memory bachane ke liye saari dictionaries pehle se download nahi karta.
# Jab yeh script pehli baar chalegi, toh yeh commands internet se stopwords ki list aur wordnet (jo lemmatization ke liye English dictionary hai) download karke layenge.
# omw-1.4 usi dictionary ka ek supporting data package hai.

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

In [ ]:

# 1. Create a small set of 10 English sentences
sentences = [
    "The quick brown fox jumps over the lazy dog.",
    "Artificial Intelligence is transforming the modern world.",
    "I genuinely love writing Python code for complex problems.",
    "She is running quickly towards the beautiful mountains.",
    "The researchers are developing new algorithms.",
    "Self-driving cars use advanced deep learning models.",
    "He laughed loudly at the hilarious joke.",
    "Data scientists analyze massive amounts of information.",
    "The cats are sleeping peacefully on the sofa.",
    "Natural language processing helps computers understand text."
]

In [ ]:
# Initialize our NLP tools
# We are using BERT's tokenizer here, which natively uses the WordPiece algorithm

# BERT = Bidirectional Encoder Representations from Transformers
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased") # uncased : case does not matter (English and english)
stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

1. The full form of BERT is Bidirectional Encoder Representations from Transformers.

"BERT Base" refers to the smaller, original version of the model, which includes 12 transformer layers and roughly 110 million parameters .

Key Aspects of BERT Base:

Bidirectional: BERT analyses text by looking at both the preceding and following words in a sentence, which helps it understand context .

Encoder: It uses only the encoder part of the transformer architecture.

Transformers: It relies on transformer architecture to process data.

Architecture Details: BERT Base has 12 Transformer layers, 12 attention heads, and a hidden state size of 768.

2. stop_words: Humne English stopwords ki list ko ek set() mein convert kar diya. Coding tip: Python mein list ke andar kisi word ko dhundhna slow hota hai, par set ke andar search karna (O(1) time complexity) bohot fast hota hai.

In [ ]:
# Let's process one sentence as an example to see the output clearly
sample_sentence = sentences[2]
print(f"--- Original Sentence ---\n{sample_sentence}\n")

--- Original Sentence ---
I genuinely love writing Python code for complex problems.



# Task 1.1: Tokenization (WordPiece)

In [ ]:
# WordPiece breaks down words into subwords if they are rare.
tokens = tokenizer.tokenize(sample_sentence)
print(f"--- 1. WordPiece Tokens ---\n{tokens}\n")

--- 1. WordPiece Tokens ---
['i', 'genuinely', 'love', 'writing', 'python', 'code', 'for', 'complex', 'problems', '.']



# Task 1.2: Stop word removal

In [ ]:
# Filtering out common words (like 'the', 'is', 'for') and keeping only alphabetic tokens
filtered_tokens = [word for word in tokens if word.lower() not in stop_words and word.isalpha()]
print(f"--- 2. After Stop Word Removal ---\n{filtered_tokens}\n")


--- 2. After Stop Word Removal ---
['genuinely', 'love', 'writing', 'python', 'code', 'complex', 'problems']



 NOTE:filtered word :  sirf alphabet ka banna chahiye and stopwords set mya nhi hona chahiye.

 here removal word : i , for

# Task 1.3: Stemming and Lemmatization

In [ ]:

print("--- 3. Stemming vs Lemmatization ---")
for word in filtered_tokens:
    stemmed_word = stemmer.stem(word)
    # Lemmatizer needs to know the context (Part of Speech), but defaults to Noun if not specified.
    # For a basic pipeline, the default is acceptable.
    lemmatized_word = lemmatizer.lemmatize(word)
    print(f"Word: {word:<12} | Stem: {stemmed_word:<10} | Lemma: {lemmatized_word}")

--- 3. Stemming vs Lemmatization ---
Word: genuinely    | Stem: genuin     | Lemma: genuinely
Word: love         | Stem: love       | Lemma: love
Word: writing      | Stem: write      | Lemma: writing
Word: python       | Stem: python     | Lemma: python
Word: code         | Stem: code       | Lemma: code
Word: complex      | Stem: complex    | Lemma: complex
Word: problems     | Stem: problem    | Lemma: problem


# Task 1.4

In [ ]:
from nltk.util import ngrams
from collections import defaultdict, Counter # defaultdictionry ager koi key nhi h .. then usko ak khali jaga de dega as a unknown word .. eroor nhi dega

In [ ]:
# Ensure punkt is downloaded for basic word tokenization
nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [ ]:
# 2. Tokenize and clean the corpus
# We convert everything to lowercase to make the n-grams case-insensitive
all_tokens = []
for sentence in sentences:
    tokens = [word.lower() for word in nltk.word_tokenize(sentence) if word.isalnum()]
    all_tokens.extend(tokens)

In [ ]:
print(all_tokens)

['the', 'quick', 'brown', 'fox', 'jumps', 'over', 'the', 'lazy', 'dog', 'artificial', 'intelligence', 'is', 'transforming', 'the', 'modern', 'world', 'i', 'genuinely', 'love', 'writing', 'python', 'code', 'for', 'complex', 'problems', 'she', 'is', 'running', 'quickly', 'towards', 'the', 'beautiful', 'mountains', 'the', 'researchers', 'are', 'developing', 'new', 'algorithms', 'cars', 'use', 'advanced', 'deep', 'learning', 'models', 'he', 'laughed', 'loudly', 'at', 'the', 'hilarious', 'joke', 'data', 'scientists', 'analyze', 'massive', 'amounts', 'of', 'information', 'the', 'cats', 'are', 'sleeping', 'peacefully', 'on', 'the', 'sofa', 'natural', 'language', 'processing', 'helps', 'computers', 'understand', 'text']


In [ ]:
# 3. Generate N-grams using NLTK
unigrams = list(ngrams(all_tokens, 1))
bigrams = list(ngrams(all_tokens, 2))
trigrams = list(ngrams(all_tokens, 3))

In [ ]:
print(unigrams)
print(bigrams)
print(trigrams)

[('the',), ('quick',), ('brown',), ('fox',), ('jumps',), ('over',), ('the',), ('lazy',), ('dog',), ('artificial',), ('intelligence',), ('is',), ('transforming',), ('the',), ('modern',), ('world',), ('i',), ('genuinely',), ('love',), ('writing',), ('python',), ('code',), ('for',), ('complex',), ('problems',), ('she',), ('is',), ('running',), ('quickly',), ('towards',), ('the',), ('beautiful',), ('mountains',), ('the',), ('researchers',), ('are',), ('developing',), ('new',), ('algorithms',), ('cars',), ('use',), ('advanced',), ('deep',), ('learning',), ('models',), ('he',), ('laughed',), ('loudly',), ('at',), ('the',), ('hilarious',), ('joke',), ('data',), ('scientists',), ('analyze',), ('massive',), ('amounts',), ('of',), ('information',), ('the',), ('cats',), ('are',), ('sleeping',), ('peacefully',), ('on',), ('the',), ('sofa',), ('natural',), ('language',), ('processing',), ('helps',), ('computers',), ('understand',), ('text',)]
[('the', 'quick'), ('quick', 'brown'), ('brown', 'fox'),

In [ ]:
# 4. Build the Language Models (Frequency Counts)
# Unigram Model: Just counts the frequency of individual words
unigram_counts = Counter(unigrams)

# Bigram Model: Maps a previous word to a Counter of possible next words
bigram_model = defaultdict(Counter)
for w1, w2 in bigrams:
    bigram_model[w1][w2] += 1

# Trigram Model: Maps a tuple of two previous words to a Counter of possible next words
trigram_model = defaultdict(Counter)
for w1, w2, w3 in trigrams:
    trigram_model[(w1, w2)][w3] += 1

In [ ]:
print(unigram_counts)

Counter({('the',): 8, ('is',): 2, ('are',): 2, ('quick',): 1, ('brown',): 1, ('fox',): 1, ('jumps',): 1, ('over',): 1, ('lazy',): 1, ('dog',): 1, ('artificial',): 1, ('intelligence',): 1, ('transforming',): 1, ('modern',): 1, ('world',): 1, ('i',): 1, ('genuinely',): 1, ('love',): 1, ('writing',): 1, ('python',): 1, ('code',): 1, ('for',): 1, ('complex',): 1, ('problems',): 1, ('she',): 1, ('running',): 1, ('quickly',): 1, ('towards',): 1, ('beautiful',): 1, ('mountains',): 1, ('researchers',): 1, ('developing',): 1, ('new',): 1, ('algorithms',): 1, ('cars',): 1, ('use',): 1, ('advanced',): 1, ('deep',): 1, ('learning',): 1, ('models',): 1, ('he',): 1, ('laughed',): 1, ('loudly',): 1, ('at',): 1, ('hilarious',): 1, ('joke',): 1, ('data',): 1, ('scientists',): 1, ('analyze',): 1, ('massive',): 1, ('amounts',): 1, ('of',): 1, ('information',): 1, ('cats',): 1, ('sleeping',): 1, ('peacefully',): 1, ('on',): 1, ('sofa',): 1, ('natural',): 1, ('language',): 1, ('processing',): 1, ('helps',)

In [ ]:
# 5. Next Word Prediction Functions
def predict_unigram():
    # Unigrams have no context. They just predict the most frequent word in the entire corpus.
    most_common = unigram_counts.most_common(1)
    return most_common[0][0][0] if most_common else None

def predict_bigram(prev_word):
    prev_word = prev_word.lower() # input word ko .lower() kiya taaki wo humare data se match ho sake.
    if prev_word in bigram_model:
        # Get the most frequent next word given the previous word
        return bigram_model[prev_word].most_common(1)[0][0]
    return "[Unknown Word]"

def predict_trigram(prev_word1, prev_word2):
    prev_word1, prev_word2 = prev_word1.lower(), prev_word2.lower()
    if (prev_word1, prev_word2) in trigram_model:
        # Get the most frequent next word given the previous two words
        return trigram_model[(prev_word1, prev_word2)].most_common(1)[0][0]
    return "[Unknown Context]"

most_common(1): Yeh Counter object ka inbuilt function hai jo sabse zyada count wale word ko list ke top par le aata hai. Example output: [('intelligence', 5)].

[0][0]: List ke pehle element [0] yaani ('intelligence', 5) ko pakda, aur us tuple ke pehle hisse [0] yaani 'intelligence' ko extract kar liya string format mein.

In [ ]:
# --- Testing the Predictions ---
print("--- Unigram Prediction ---")
print(f"Prediction (No context): {predict_unigram()}")

print("\n--- Bigram Prediction ---")
test_word = "artificial"
print(f"Input: '{test_word}' -> Predicted Next: '{predict_bigram(test_word)}'")
test_word_2 = "the"
print(f"Input: '{test_word_2}' -> Predicted Next: '{predict_bigram(test_word_2)}'")

print("\n--- Trigram Prediction ---")
w1, w2 = "deep", "learning"
print(f"Input: '{w1} {w2}' -> Predicted Next: '{predict_trigram(w1, w2)}'")

--- Unigram Prediction ---
Prediction (No context): the

--- Bigram Prediction ---
Input: 'artificial' -> Predicted Next: 'intelligence'
Input: 'the' -> Predicted Next: 'quick'

--- Trigram Prediction ---
Input: 'deep learning' -> Predicted Next: 'models'


# Task 1.5

In [ ]:
!pip install editdistance

> Edit distance is a fundamental concept in Natural Language Processing (NLP) used to quantify the dissimilarity between two strings (like words or sentences) by measuring the minimum number of editing operations required to transform one into the other.

In [ ]:
import editdistance
from nltk.util import ngrams

In [ ]:
# Task 1.5.3: Build vocabulary
vocabulary = set(all_tokens)

In [ ]:
# Build Unigram and Bigram counts for probability calculations
unigram_counts = Counter(all_tokens)
bigram_counts = defaultdict(Counter)

for w1, w2 in list(ngrams(all_tokens, 2)):
    bigram_counts[w1][w2] += 1

def get_bigram_probability(prev_word, current_word):
    """Calculates P(current_word | prev_word)"""
    if unigram_counts[prev_word] == 0:
        return 0.0
    # Count of (prev_word -> current_word) / Count of (prev_word)
    return bigram_counts[prev_word][current_word] / unigram_counts[prev_word]

In [ ]:
# Task 1.5.4: Take a new sentence from the user
# I've intentionally misspelled "brown" as "brawn" and "cats" as "cots"
user_sentence = "The brawn fox and the cots"
user_tokens = [word.lower() for word in nltk.word_tokenize(user_sentence) if word.isalnum()]

print(f"Original User Sentence: '{user_sentence}'\n")


Original User Sentence: 'The brawn fox and the cots'



In [ ]:
# Task 1.5.5: Iterate over each word
corrected_sentence = []

for i, selected_word in enumerate(user_tokens):

    # Find candidate word from vocab with LEAST Levenshtein distance
    min_distance = float('inf')
    candidate_word = selected_word

    for vocab_word in vocabulary:
        dist = editdistance.eval(selected_word, vocab_word)
        if dist < min_distance:
            min_distance = dist
            candidate_word = vocab_word

    # Task 1.5.6 & 1.5.7: Use Bigram model to decide which word is more likely
    if i == 0:
        # If it's the first word, we have no previous word context.
        # We just pick the candidate with the lowest edit distance.
        final_word = candidate_word
    else:
        # Get the previous word (we use the *corrected* previous word for better context)
        prev_word = corrected_sentence[i-1]

        # Calculate probabilities
        prob_selected = get_bigram_probability(prev_word, selected_word)
        prob_candidate = get_bigram_probability(prev_word, candidate_word)

        print(f"Checking '{selected_word}':")
        print(f"  Candidate found: '{candidate_word}' (Edit Dist: {min_distance})")
        print(f"  P({selected_word} | {prev_word}) = {prob_selected:.4f}")
        print(f"  P({candidate_word} | {prev_word}) = {prob_candidate:.4f}")

        # Compare probabilities
        if prob_candidate > prob_selected:
            final_word = candidate_word
            print(f"  -> Action: Replaced '{selected_word}' with '{candidate_word}'")
        elif prob_selected > prob_candidate:
            final_word = selected_word
            print(f"  -> Action: Kept original '{selected_word}'")
        else:
            # If probabilities are equal (e.g., both 0), trust the spell checker's candidate
            final_word = candidate_word if selected_word not in vocabulary else selected_word
            print(f"  -> Action: Probabilities tied. Defaulting to '{final_word}'")

    corrected_sentence.append(final_word)
    print("-" * 40)

print(f"\nFinal Corrected Sentence: '{' '.join(corrected_sentence)}'")

----------------------------------------
Checking 'brawn':
  Candidate found: 'brown' (Edit Dist: 1)
  P(brawn | the) = 0.0000
  P(brown | the) = 0.0000
  -> Action: Probabilities tied. Defaulting to 'brown'
----------------------------------------
Checking 'fox':
  Candidate found: 'fox' (Edit Dist: 0)
  P(fox | brown) = 1.0000
  P(fox | brown) = 1.0000
  -> Action: Probabilities tied. Defaulting to 'fox'
----------------------------------------
Checking 'and':
  Candidate found: 'are' (Edit Dist: 2)
  P(and | fox) = 0.0000
  P(are | fox) = 0.0000
  -> Action: Probabilities tied. Defaulting to 'are'
----------------------------------------
Checking 'the':
  Candidate found: 'the' (Edit Dist: 0)
  P(the | are) = 0.0000
  P(the | are) = 0.0000
  -> Action: Probabilities tied. Defaulting to 'the'
----------------------------------------
Checking 'cots':
  Candidate found: 'cats' (Edit Dist: 1)
  P(cots | the) = 0.0000
  P(cats | the) = 0.1250
  -> Action: Replaced 'cots' with 'cats'
----

> Note:  Levenshtein Distance Check (Finding Candidates)

Kyun kiya? Agar user ne "brawn" type kiya hai, toh hume vocabulary me se sabse closely matching word dhoondhna hai (jaise "brown").

float('inf'): Hum minimum distance dhoondh rahe hain, isliye shuruwaati base value infinity rakhte hain.

Loop har ek vocabulary word ke sath user ke word ko compare karta hai. editdistance.eval() batata hai ki kitne letters change karne padenge.
Jiska distance sabse kam hota hai, usko hum candidate_word maan lete hain.

> Note: Context Check (The Smart Decision)

>> Sirf spelling check karna kaafi nahi hai. Agar user ne "cots" likha hai, toh uska edit distance "cats" aur "coats" dono se 1 ho sakta hai. Sahi word kaunsa hai, yeh pichle word (context) se pata chalega.

> Note :   if i == 0: Agar sentence ka pehla hi word hai, toh uske peeche koi context nahi hai. Isliye hum chup-chaap lowest edit distance wale word ko final maan lete hain.

>> else:  Hum calculate karte hain:

>>> prob_selected: User ne jo word type kiya hai ("cots"), uska pichle word ("the") ke baad aane ka kya chance hai? (Most likely 0.0)

>>prob_candidate: Model ne jo word suggest kiya hai ("cats"), uska aane ka kya chance hai? (High chance, kyunki humne "the cats" corpus mein train kiya hai).

>>if prob_candidate > prob_selected: Agar suggest kiye hue word ki probability zyada logical lag rahi hai, toh user ke word ko replace kar do!